In [ ]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import myfunction as mf
path_data_raw = "C:/Users/dell/OneDrive/file/"
path_country_nc = "C:/Users/dell/OneDrive/file/nc"
path_one_spdb = 'C:/Users/dell/OneDrive/file/SPDB/'
drive_letter = 'E:'

path_pre = drive_letter + "/wyy/code_project/running_outcome/final_data/SPDB/part0_treat/pretreatment/"
path_match = drive_letter + "/wyy/code_project/running_outcome/final_data/SPDB/part0_treat/match/"
path_var_data = drive_letter + "/wyy/SPDB_database/data/raw/"
path_match_geo = drive_letter + "/wyy/code_project/running_outcome/final_data/SPDB/part0_treat/match/geo/"




meta_name = "meta_data.csv"
mark_num = "25"
list_color = ["#ee877c", "#8bd0e3", "#6abeae", "#808eaf", "#f7bba8", "#acb4cc", "#b5e0d5", "#e86462", "#a89687"]

In [ ]:


def count_variance(df_sem_data):
    """计算所有变量的方法  返回的是一个df"""
    print(df_sem_data.shape)
    mean = df_sem_data.mean()
    std_dev = df_sem_data.std()
    variance = df_sem_data.var()


    df_results = pd.DataFrame({
        'var': df_sem_data.columns,
        'mean': mean.values,
        'SD': std_dev.values,
        'variance': variance.values
    })


    min_variance = df_results["variance"].min()
    max_variance = df_results["variance"].max()
    print(min_variance, max_variance)
    print(max_variance/min_variance)

    return df_results

def convert_temp(df_o, from_scale, to_scale):

    df = df_o.copy()
    df_meta = pd.read_csv(path_data_raw + meta_name, encoding="utf-8")
    list_all_var = df_meta['var_name'][(df_meta['var_select' + mark_num]==1)&(df_meta['var_temp']==1)].to_list()
    for col in list_all_var:
        if from_scale == "C":
            if to_scale == "F":
                df[col] = (df[col] * 9/5) + 32
            elif to_scale == "K":
                df[col] = df[col] + 273.15
        elif from_scale == "F":
            if to_scale == "C":
                df[col] = (df[col] - 32) * 5/9
            elif to_scale == "K":
                df[col] = ((df[col] - 32) * 5/9) + 273.15
        elif from_scale == "K":
            if to_scale == "C":
                df[col] = df[col] - 273.15
            elif to_scale == "F":
                df[col] = ((df[col] - 273.15) * 9/5) + 32
    return df

### geo raw
geo这边是一个个分散的nc文件  
这种形式的数据不方便调用  
这里把他们全部合并到一个csv文件中  

In [ ]:


import pandas as pd
import numpy as np


lat_range = np.arange(-90, 90, 1)
lon_range = np.arange(-180, 180, 1)
year_range = np.arange(2000, 2021, 1)


df1 = pd.DataFrame({
    'lat_grid': np.repeat(lat_range, len(lon_range) * len(year_range)),
    'lon_grid': np.tile(np.repeat(lon_range, len(year_range)), len(lat_range)),
    'year': np.tile(year_range, len(lat_range) * len(lon_range))
})


df2 = pd.DataFrame({
    'lat_grid': np.repeat(lat_range, len(lon_range)),
    'lon_grid': np.tile(lon_range, len(lat_range))
})

print(df1.shape)
print(df2.shape)

df1.to_csv(path_match + "lat_lon_year.csv", index=False)
df2.to_csv(path_match + "lat_lon.csv", index=False)


In [ ]:

import os
import pandas as pd
import numpy as np
import xarray as xr




int_grid = 1
data_type = "raw"







nc_dir_year = path_var_data + str(int_grid) + "\\year"
nc_dir_one = path_var_data + str(int_grid) + "\\one"


df_meta = pd.read_csv(path_data_raw + meta_name, encoding="utf-8")
list_var = df_meta['var_name'][(df_meta['file_type']=='nc')&(df_meta['var_select' + mark_num]==1)].tolist()
list_var = [var + "_" + str(int_grid) + "x" + str(int_grid) + ".nc" for var in list_var]


def dataset_to_dataframe(ds, var_name, index_vars):
    """
    Convert an xarray Dataset to a pandas DataFrame.

    :param ds: xarray Dataset to convert
    :param var_name: the variable name to extract from the Dataset
    :param index_vars: a list of variable names to use as index in the DataFrame
    :return: pandas DataFrame with the specified variable and index
    """
    df = ds[var_name].to_dataframe().reset_index()
    df = df.set_index(index_vars)
    return df

def process_nc_dir_one(nc_dir_one, list_var, df_geo, int_grid):
    for filename in os.listdir(nc_dir_one):
        if filename in list_var:

            with xr.open_dataset(os.path.join(nc_dir_one, filename)) as ds:
                var_name = [var for var in ds.data_vars][0]
                column_name = filename.replace("_" + str(int_grid) + "x" + str(int_grid) + ".nc", "")


                df_nc = dataset_to_dataframe(ds, var_name, ['lon', 'lat'])


                df_geo = df_geo.merge(df_nc, left_on=['lon_grid', 'lat_grid'], right_index=True, how='left')
                df_geo = df_geo.rename(columns={var_name: column_name})
    return df_geo

def process_nc_dir_year(nc_dir_year, list_var, df_geo, int_grid):
    for filename in os.listdir(nc_dir_year):
        if filename in list_var:

            with xr.open_dataset(os.path.join(nc_dir_year, filename)) as ds:
                ds = ds.sortby('lon')
                ds = ds.sortby('lat')
                ds = ds.sortby('year')
                var_name = [var for var in ds.data_vars][0]
                column_name = filename.replace("_" + str(int_grid) + "x" + str(int_grid) + ".nc", "")


                df_nc = dataset_to_dataframe(ds, var_name, ['lon', 'lat', 'year'])


                df_geo = df_geo.merge(df_nc, left_on=['lon_grid', 'lat_grid', 'year'], right_index=True, how='left')
                df_geo = df_geo.rename(columns={var_name: column_name})
    return df_geo


for year in range(2000, 2021):
    print(year)
    df_geo = pd.read_csv(path_match + "lat_lon.csv")
    df_geo = df_geo.drop_duplicates(subset=['lat_grid', 'lon_grid'])
    df_geo['year'] = year

    print(df_geo.columns)
    print(df_geo.shape)

    df_geo = process_nc_dir_one(nc_dir_one,list_var, df_geo, int_grid)
    df_geo = process_nc_dir_year(nc_dir_year,list_var, df_geo, int_grid)



    if data_type == "normalization" or data_type == "standardization":
        for column in df_geo.columns:
            if column not in ['lat_grid', 'lon_grid', 'year', 'country_id']:
                if df_geo[column].lt(0).any() or df_geo[column].gt(1).any() or df_geo[column].lt(1e-20).any():
                    print(f'Column {column} has values out of range [0, 1] or less than 1e-20. Replacing these values with 0.')
                    df_geo.loc[df_geo[column].lt(0) | df_geo[column].gt(1) | df_geo[column].lt(1e-20), column] = 0



    df_geo.fillna(0, inplace=True)

    def replace_nan_in_array(array):
        return np.where(np.isnan(array), 0, array)

    df_geo2 = df_geo.applymap(replace_nan_in_array)
    df_geo2 = df_geo2.fillna(0)
    df_geo2 = df_geo2.replace(["nan", "NaN", "NAN", "", None], 0)


    column_means = df_geo2.mean()


    filtered_columns = column_means[(column_means > 1) | (column_means == 0)]


    columns_list = filtered_columns.index.tolist()


    print(columns_list)


    df_geo2.to_csv(path_match_geo + 'geo_global_'+str(year)+'.csv', encoding='utf-8-sig', index=False)



In [ ]:

import os
import pandas as pd


folder_path = r'E:\wyy\code_project\running_outcome\final_data\SPDB\part0_treat\match\geo'


file_names = [file for file in os.listdir(folder_path) if file.endswith('.csv')]


dfs = [pd.read_csv(os.path.join(folder_path, file)) for file in file_names]
merged_df = pd.concat(dfs, ignore_index=True)


output_file = r'E:\wyy\code_project\running_outcome\final_data\SPDB\part0_treat\match\geo_global.csv'


merged_df.to_csv(output_file, index=False)

print("合并完成并已保存到指定路径下。")


In [ ]:


df_data_raw = pd.read_csv(path_match + 'geo_global.csv')

df_data_raw = df_data_raw[(df_data_raw['lon_grid'] != -180) & (df_data_raw['lat_grid'] != -90)]

df_meta = pd.read_csv(path_data_raw + meta_name, encoding="utf-8")
df_data_raw = convert_temp(df_data_raw, "K", "C")

list_meta = df_meta["var_name"][df_meta["var_select" + mark_num]==1].to_list()
list_data = df_data_raw.columns.tolist()
list_select = list(set(list_meta)&set(list_data))

df_data1 = df_data_raw[list_select]

second_treat = "auto"

if second_treat == "N":
    pass
else:
    df_data1 = mf.preprocess(df_data1, treat_method=second_treat)


feature_type = "standardization"
df_data1 = mf.preprocess(df_data1, treat_method=feature_type)
df_sem_variance = count_variance(df_data1)


treat_mark1 = "au_"
treat_mark2 = "to_"

df_sem_variance.to_csv(path_match_geo + "variance_geo"+mark_num+".csv", index=False)

df_data = pd.concat([df_data1, df_data_raw[["lon_grid", "lat_grid", 'year']]], axis=1)
sem_data_name = "geo_global_"+ treat_mark1 + treat_mark2 + mark_num+".csv"
print(sem_data_name)
print(df_data.columns)
df_data.to_csv(path_match_geo + sem_data_name, index=False)